In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor

In [2]:
from google.colab import files
uploaded = files.upload()


Saving Train.csv to Train.csv


In [3]:
df = pd.read_csv("Train.csv")

In [4]:
print("Dataset Shape:",df.shape)
df.head()

Dataset Shape: (10999, 12)


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


In [5]:
print(df.isnull().sum())

ID                     0
Warehouse_block        0
Mode_of_Shipment       0
Customer_care_calls    0
Customer_rating        0
Cost_of_the_Product    0
Prior_purchases        0
Product_importance     0
Gender                 0
Discount_offered       0
Weight_in_gms          0
Reached.on.Time_Y.N    0
dtype: int64


In [6]:
X = df.drop("Cost_of_the_Product", axis=1)
y = df["Cost_of_the_Product"]

In [7]:
X_encoded = X.copy()
cat_cols = X_encoded.select_dtypes(include='object').columns
for col in cat_cols:
    X_encoded[col] = LabelEncoder().fit_transform(X_encoded[col])
X_encoded.head()

,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,3,0,4,2,3,1,0,44,1233,1
1,2,4,0,4,5,2,1,1,59,3088,1
2,3,0,0,2,2,4,1,1,48,3374,1
3,4,1,0,3,3,4,2,1,10,1177,1
4,5,2,0,2,2,3,2,0,46,2484,1


In [8]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

In [15]:
pca_pipeline = Pipeline([
    ("pca", PCA(n_components=2))
])

In [16]:
kbest_pipeline = Pipeline([
    ("select_best", SelectKBest(score_func=f_classif, k=2))
])

In [17]:
combined_features = FeatureUnion([
    ("pca_pipeline", pca_pipeline),
    ("kbest_pipeline", kbest_pipeline)
])

X_combined = combined_features.fit_transform(
    X_scaled,
    df["Reached.on.Time_Y.N"]
)

print("Feature Union Shape:", X_combined.shape)

Feature Union Shape: (10999, 4)


In [18]:
model = TransformedTargetRegressor(
    regressor=LinearRegression(),
    func=np.log1p,
    inverse_func=np.expm1
)

In [19]:
model.fit(X_combined, y)
print("Regression model trained")

Regression model trained


In [20]:
prediction = model.predict(X_combined[:5])
print("Sample Predictions:")
print(prediction)

Sample Predictions:
[192.65538107 181.37775352 177.28578864 197.06023031 177.57347651]
